# Musahhih: untouched Gemma 3 baseline on Nahw-Passage

Musahhih studies Modern Standard Arabic grammatical error correction with open-weight models. This notebook measures an **untouched-model baseline**: it asks an instruction model to correct one identified erroneous word in each passage.

**Research boundary:** `Nahw-Passage` is strictly test-only. This notebook does not train, fine-tune, attach LoRA adapters, modify model weights, tune the prompt on test results, or select a checkpoint. The default run evaluates exactly the first 25 prepared records as a pipeline pilot. Do not treat that pilot as a development set.

## 1. Runtime and GPU verification

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run this cell again. Free GPU availability is not guaranteed and Colab Pro is not required.

In [ ]:
import platform
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is assigned. In Colab select Runtime > Change runtime type > "
        "T4 GPU, save, and rerun from the top. Free GPU allocation is subject to availability."
    )

GPU_NAME = torch.cuda.get_device_name(0)
gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info(0)
GPU_FREE_MEMORY_GIB = gpu_free_bytes / 1024**3
GPU_MEMORY_GIB = gpu_total_bytes / 1024**3
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime reported by PyTorch: {torch.version.cuda}")
print(f"GPU: {GPU_NAME}")
print(f"GPU memory available: {GPU_FREE_MEMORY_GIB:.2f} GiB")
print(f"GPU memory total: {GPU_MEMORY_GIB:.2f} GiB")

## 2. Repository setup

This cell clones the official Musahhih repository when absent. On reruns it performs a fast-forward-only update only when the checkout is clean; it never deletes or overwrites local work.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ALBA7OOTH-Research-Lab/Musahhih.git"
current = Path.cwd().resolve()
if (current / "AGENTS.md").exists() and (current / ".git").exists():
    repo_dir = current
else:
    repo_dir = Path("/content/Musahhih") if Path("/content").exists() else current / "Musahhih"

if (repo_dir / ".git").exists():
    dirty = subprocess.run(
        ["git", "-C", str(repo_dir), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        print("Existing checkout has local changes; skipping pull to preserve them.")
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
elif repo_dir.exists():
    raise RuntimeError(f"{repo_dir} exists but is not a Git checkout; refusing to delete or replace it.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

os.chdir(repo_dir)
print(f"Repository: {Path.cwd()}")

## 3. Install the supported Unsloth stack

The commands below follow [Unsloth's official Gemma 3 4B Colab notebook](https://github.com/unslothai/notebooks/blob/main/nb/Gemma3_(4B).ipynb): they match Colab's installed PyTorch to a compatible `xformers`, then pin the model-specific `transformers`, `trl`, and `datasets` versions used by that workflow. Rerunning is safe but may still take several minutes. Restart the runtime only if pip explicitly asks you to.

In [ ]:
import re

torch_line = re.match(r"[0-9]+\.[0-9]+", torch.__version__)
torch_minor = torch_line.group(0) if torch_line else ""
xformers_version = {
    "2.10": "0.0.34",
    "2.9": "0.0.33.post1",
    "2.8": "0.0.32.post2",
}.get(torch_minor, "0.0.34")

def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

pip_install("sentencepiece", "protobuf", "datasets==4.3.0", "huggingface_hub>=0.34.0", "hf_transfer")
pip_install("--no-deps", "unsloth_zoo", "bitsandbytes", "accelerate", f"xformers=={xformers_version}", "peft", "trl", "triton", "unsloth")
pip_install("--no-deps", "--upgrade", "torchao>=0.16.0")
pip_install("transformers==4.56.2")
pip_install("--no-deps", "trl==0.22.2")
print("Dependency installation complete.")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PACKAGE_NAMES = ["torch", "transformers", "unsloth", "accelerate", "peft", "trl", "datasets", "bitsandbytes"]
PACKAGE_VERSIONS = {}
for package in PACKAGE_NAMES:
    try:
        PACKAGE_VERSIONS[package] = version(package)
    except PackageNotFoundError:
        PACKAGE_VERSIONS[package] = None

RUNTIME_INFO = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": GPU_NAME,
    "gpu_memory_available_gib_at_check": round(GPU_FREE_MEMORY_GIB, 2),
    "gpu_memory_total_gib": round(GPU_MEMORY_GIB, 2),
}
print("Packages:", PACKAGE_VERSIONS)
print("Runtime:", RUNTIME_INFO)

## 4. Prepare and verify Nahw-Passage

The repository scripts download from `https://github.com/qcri/nahw-arabic-grammar-benchmark`, inspect the official files, and create a UTF-8 evaluation JSONL. Nothing in this stage creates training data.

In [ ]:
for script in ["download_nahw.py", "inspect_nahw.py", "prepare_nahw_eval.py"]:
    subprocess.run([sys.executable, f"scripts/{script}"], check=True)

In [ ]:
import hashlib
import json

RAW_TEST_PATH = Path("data/raw/nahw/Nahw-Passage.json")
EVAL_PATH = Path("data/processed/nahw_gec_test.jsonl")
assert RAW_TEST_PATH.is_file(), "Official Nahw-Passage.json was not downloaded."
assert EVAL_PATH.is_file(), "Prepared evaluation JSONL was not created."

with RAW_TEST_PATH.open("r", encoding="utf-8") as handle:
    raw_records = json.load(handle)
with EVAL_PATH.open("r", encoding="utf-8") as handle:
    eval_records = [json.loads(line) for line in handle if line.strip()]

required_fields = {"id", "passage_id", "prompt", "passage", "error", "gold_correction", "split", "source"}
assert len(raw_records) == 511, f"Expected 511 official correction records, found {len(raw_records)}"
assert len(eval_records) == 511, f"Expected 511 prepared correction records, found {len(eval_records)}"
assert all(required_fields <= record.keys() for record in eval_records), "A prepared record is missing required fields."
assert all(record["split"] == "test" for record in eval_records), "Every record must be test-only."
assert all(record["source"] == "Nahw-Passage" for record in eval_records), "Unexpected data source."
TEST_FILE_SHA256 = hashlib.sha256(EVAL_PATH.read_bytes()).hexdigest()
print(f"Verified {len(eval_records)} UTF-8 Nahw-Passage correction records (test-only).")
print(f"Prepared test SHA256: {TEST_FILE_SHA256}")

## 5. Load the untouched 4-bit model

We use `unsloth/gemma-3-4b-it-unsloth-bnb-4bit`, Unsloth's public 4-bit quantization of `google/gemma-3-4b-it`. This keeps the architecture directly comparable with Nahw while avoiding the gated Google repository. Gemma 3 uses Unsloth's `FastModel` multimodal loader even though this task supplies text only. No adapter is attached and no training API is called.

A free Colab T4 normally provides about 15 GiB VRAM; a 4B model in 4-bit is expected to fit for short inference, but Colab images and memory allocations change. The recorded model revision and runtime metadata are authoritative for each run.

In [ ]:
from huggingface_hub import model_info
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

MODEL_ID = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
MAX_SEQUENCE_LENGTH = 2048
MODEL_REVISION = model_info(MODEL_ID).sha

model, processor = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
processor = get_chat_template(processor, chat_template="gemma-3")
model.eval()
MODEL_LOAD_INFO = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "processor_class": processor.__class__.__name__,
    "processor_name_or_path": getattr(processor, "name_or_path", None),
    "quantization": "bitsandbytes 4-bit via Unsloth pre-quantized checkpoint",
    "load_in_4bit": True,
    "load_in_8bit": False,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "dtype": str(getattr(model, "dtype", None)),
    "gpu": GPU_NAME,
    "lora_adapters_attached": False,
}
print(json.dumps(MODEL_LOAD_INFO, indent=2))

## 6. Fixed prompt and conservative parser

The prompt is fixed by `scripts/prepare_nahw_eval.py`: it shows the passage and identified erroneous word, then requests only the corrected word. Do not revise it after inspecting these test outputs.

The parser removes only outer whitespace/empty lines, paired quotation marks, and paired Markdown wrappers. It does **not** normalize Arabic letters, hamza forms, diacritics, Unicode composition, or internal punctuation; it never sees the gold answer. Raw responses are always retained.

In [ ]:
from scripts.nahw_baseline_utils import parse_model_response, summarize_predictions
from scripts.prepare_nahw_eval import PROMPT as PROMPT_TEMPLATE

assert all(
    record["prompt"] == PROMPT_TEMPLATE.format(passage=record["passage"], error=record["error"])
    for record in eval_records
)
assert parse_model_response(" **«مُصَحَّحة»** " )[0] == "مُصَحَّحة"
assert parse_model_response("إجابة،أخرى")[0] == "إجابة،أخرى"
assert parse_model_response("إجابة\nشرح")[0] == "إجابة\nشرح"
print(PROMPT_TEMPLATE)

## 7. Deterministic inference and result writing

Generation uses greedy decoding (`do_sample=False`). Temperature is not passed because Transformers ignores sampling temperature during greedy decoding. `max_new_tokens` is recorded explicitly.

In [ ]:
from datetime import datetime, timezone
from tqdm.auto import tqdm
from transformers import set_seed

RANDOM_SEED = 3407
set_seed(RANDOM_SEED)
DECODING_SETTINGS = {
    "seed": RANDOM_SEED,
    "do_sample": False,
    "temperature": None,
    "temperature_behavior": "not passed; greedy decoding is deterministic",
    "max_new_tokens": 32,
}

def git_commit_sha():
    result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None

def generate_response(full_prompt):
    messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.inference_mode():
        generated = model.generate(
            **inputs, do_sample=False, max_new_tokens=DECODING_SETTINGS["max_new_tokens"],
        )
    new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True)

def run_records(records, predictions_path, summary_path, run_label):
    predictions = []
    for record in tqdm(records, desc=run_label):
        raw_response = generate_response(record["prompt"])
        parsed, warnings = parse_model_response(raw_response)
        normalized_gold = record["gold_correction"].strip()  # whitespace only; no Arabic normalization
        predictions.append({
            "record_id": record["id"],
            "passage_id": record["passage_id"],
            "full_passage": record["passage"],
            "erroneous_word": record["error"],
            "gold_correction": record["gold_correction"],
            "full_prompt": record["prompt"],
            "raw_model_response": raw_response,
            "parsed_correction": parsed,
            "normalized_gold_value": normalized_gold,
            "exact_match": parsed == normalized_gold,
            "parsing_warning": "; ".join(warnings) if warnings else None,
            "parsing_warnings": warnings,
            "source": record["source"],
            "split": record["split"],
        })

    predictions_path = Path(predictions_path)
    summary_path = Path(summary_path)
    predictions_path.parent.mkdir(parents=True, exist_ok=True)
    with predictions_path.open("w", encoding="utf-8", newline="\n") as handle:
        for prediction in predictions:
            handle.write(json.dumps(prediction, ensure_ascii=False) + "\n")

    summary = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "run_label": run_label,
        **MODEL_LOAD_INFO,
        "package_versions": PACKAGE_VERSIONS,
        "runtime": RUNTIME_INFO,
        **summarize_predictions(predictions),
        "prompt_template": PROMPT_TEMPLATE,
        "decoding_settings": DECODING_SETTINGS,
        "test_file": str(EVAL_PATH),
        "test_file_sha256": TEST_FILE_SHA256,
        "git_commit_sha": git_commit_sha(),
        "predictions_file": str(predictions_path),
        "metric_definitions": {
            "normalized_gold_value": "outer whitespace stripped only; Arabic text otherwise unchanged",
            "parsing_failure_count": "empty parsed responses",
            "suspicious_output_count": "responses with warnings other than harmless outer-format cleanup",
        },
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return predictions, summary

## 8. Run the 25-record pilot

This is the default and intentionally fixed at exactly 25 test records. It validates the mechanics, not prompt quality. Do not alter the prompt based on the resulting errors.

In [ ]:
PILOT_RECORD_COUNT = 25
pilot_records = eval_records[:PILOT_RECORD_COUNT]
assert len(pilot_records) == 25
pilot_predictions, pilot_summary = run_records(
    pilot_records,
    "outputs/baseline_pilot_predictions.jsonl",
    "outputs/baseline_pilot_summary.json",
    "nahw_passage_pilot_25",
)
print(json.dumps(pilot_summary, ensure_ascii=False, indent=2))

## 9. Manual review

Inspect all 25 raw/parsed outcomes before considering the optional full run. A warning is diagnostic, not a linguistic judgment.

In [ ]:
import pandas as pd

review_table = pd.DataFrame([{
    "erroneous word": row["erroneous_word"],
    "gold correction": row["gold_correction"],
    "prediction": row["parsed_correction"],
    "exact match": row["exact_match"],
    "parsing warning": row["parsing_warning"],
} for row in pilot_predictions])
assert len(review_table) == 25
display(review_table)

## 10. Optional full 511-record benchmark — disabled by default

Do not enable this until the model loads, the parser preserves the intended text, all 25 pilot outputs have been manually reviewed, and the scoring logic has been checked. Enabling a full run is a deliberate evaluation action, not an opportunity to tune the prompt. Set `RUN_FULL_BENCHMARK = True` only after those checks. Full outputs use separate filenames.

In [ ]:
RUN_FULL_BENCHMARK = False  # Deliberately change to True only after the checklist above.

if RUN_FULL_BENCHMARK:
    assert len(eval_records) == 511
    full_predictions, full_summary = run_records(
        eval_records,
        "outputs/baseline_full_predictions.jsonl",
        "outputs/baseline_full_summary.json",
        "nahw_passage_full_511",
    )
    print(json.dumps(full_summary, ensure_ascii=False, indent=2))
else:
    print("Full benchmark disabled. The default notebook run remains the 25-record pilot.")

## 11. Optional export

The first cell downloads existing result files through Colab. The second optionally mounts Google Drive and copies results. Neither writes credentials into this notebook, and Drive is not required.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    for result_path in sorted(Path("outputs").glob("baseline_*.*")):
        files.download(str(result_path))
else:
    print("Set DOWNLOAD_RESULTS = True to download result files from Colab.")

In [ ]:
COPY_RESULTS_TO_DRIVE = False
if COPY_RESULTS_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/Musahhih/results")
    destination.mkdir(parents=True, exist_ok=True)
    for result_path in sorted(Path("outputs").glob("baseline_*.*")):
        shutil.copy2(result_path, destination / result_path.name)
    print(f"Copied results to {destination}")
else:
    print("Google Drive is optional. Set COPY_RESULTS_TO_DRIVE = True to mount and copy results.")